# Data Preparation


## Load Data

In [94]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from sklearn.cluster import KMeans
import seaborn as sns
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans
server = r"SIX\SQLEXPRESS"
database = "OlistDB"

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

# Load các bảng chính (điều chỉnh path theo thư mục của bạn)
dataset_dir = r'C:\Users\nghah\OneDrive\Documents\dataset'

# Đọc các file CSV bằng cách nối thư mục gốc với tên file
orders = pd.read_csv(os.path.join(dataset_dir, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(dataset_dir, 'olist_customers_dataset.csv'))
order_items = pd.read_csv(os.path.join(dataset_dir, 'olist_order_items_dataset.csv'))
payments = pd.read_csv(os.path.join(dataset_dir, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(dataset_dir, 'olist_order_reviews_dataset.csv'))
products = pd.read_csv(os.path.join(dataset_dir, 'olist_products_dataset.csv'))
sellers = pd.read_csv(os.path.join(dataset_dir, 'olist_sellers_dataset.csv'))
category = pd.read_csv(os.path.join(dataset_dir, 'product_category_name_translation.csv'))

# Convert timestamp columns sang datetime ngay từ đầu
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

In [1]:
%run "clean.ipynb"

NULL trước khi xử lý:
product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64
product_weight_g: điền 0 NULL còn lại bằng median = 700.0
product_length_cm: điền 0 NULL còn lại bằng median = 25.0
product_height_cm: điền 0 NULL còn lại bằng median = 13.0
product_width_cm: điền 0 NULL còn lại bằng median = 20.0

NULL sau khi xử lý:
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64
--- TRƯỚC KHI XỬ LÝ OUTLIER ---
count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64

Q1=39.90, Q3=134.90, IQR=95.00, upper_bound=277.40
Số dòng vượt ngưỡng IQR: 8427
Số dòng cực đoan (>5000 BRL): 3

--- SAU KHI XỬ LÝ OUTLIER ---
count    112647.000000
mean         98.439457
std          75.918638
min           0.850000
25%          39.900000
50%         

## Preparing

### Classification payment

In [13]:
payment_encoded = pd.get_dummies(payments['payment_type'], prefix='payment')

print("--- KIỂM CHỨNG ONE-HOT ---")
print(payment_encoded.sum())   # đếm số lượng mỗi loại
print("\nTỷ lệ %:")
print((payment_encoded.sum() / len(payment_encoded) * 100).round(2))

--- KIỂM CHỨNG ONE-HOT ---
payment_boleto         19784
payment_credit_card    76795
payment_debit_card      1529
payment_not_defined        3
payment_voucher         5775
dtype: int64

Tỷ lệ %:
payment_boleto         19.04
payment_credit_card    73.92
payment_debit_card      1.47
payment_not_defined     0.00
payment_voucher         5.56
dtype: float64


### label customers state

In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
customers['customer_state_encoded'] = le.fit_transform(customers['customer_state'])

print("--- KIỂM CHỨNG LABEL ENCODING ---")
print(f"Số lượng nhãn (state) duy nhất: {len(le.classes_)}")
print(f"Khoảng giá trị mã hóa: {customers['customer_state_encoded'].min()} - {customers['customer_state_encoded'].max()}")
print("\nBảng ánh xạ mẫu:")
for state, code in list(zip(le.classes_, range(len(le.classes_))))[:5]:
    print(f"  {state} -> {code}")

--- KIỂM CHỨNG LABEL ENCODING ---
Số lượng nhãn (state) duy nhất: 27
Khoảng giá trị mã hóa: 0 - 26

Bảng ánh xạ mẫu:
  AC -> 0
  AL -> 1
  AM -> 2
  AP -> 3
  BA -> 4


In [15]:
order_value_df = (
    order_items
    .groupby('order_id', as_index=False)
    .agg(order_total=('price', 'sum'))
)

order_value_df.head()

,order_id,order_total
0,00010242fe8c5a6d1ba2dd792cb16214,58.90
1,00018f77f2f0320c557190d7a144bdd3,239.90
2,000229ec398224ef6ca0657da4fc703e,199.00
3,00024acbcdf0a6daa1e931b038114c75,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90


In [16]:
import pandas as pd

# ============================================================
# BƯỚC 1: Merge delivered (orders đã lọc status = delivered) 
# với customers để lấy customer_unique_id
# ============================================================
orders_customers_df = delivered.merge(
    customers[['customer_id', 'customer_unique_id']],
    on='customer_id',
    how='left'
)

# ============================================================
# BƯỚC 2: Merge với order_value_df (đã có sẵn order_total)
# ============================================================
orders_full_df = orders_customers_df.merge(
    order_value_df[['order_id', 'order_total']],   # đổi tên cột nếu khác
    on='order_id',
    how='left'
)

# Đảm bảo timestamp đúng kiểu datetime
orders_full_df['order_purchase_timestamp'] = pd.to_datetime(
    orders_full_df['order_purchase_timestamp']
)

# ============================================================
# BƯỚC 3: Snapshot date để tính Recency
# ============================================================
snapshot_date = orders_full_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# ============================================================
# BƯỚC 4: Tính RFM theo customer_unique_id
# ============================================================
rfm_df = (
    orders_full_df
    .groupby('customer_unique_id')
    .agg(
        recency_days=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
        purchase_frequency=('order_id', 'nunique'),
        monetary_total=('order_total', 'sum')
    )
    .reset_index()
)

rfm_df = rfm_df.dropna(subset=['monetary_total'])

print("--- KIỂM CHỨNG RFM ---")
print(f"Số khách hàng duy nhất (customer_unique_id): {rfm_df['customer_unique_id'].nunique()}")
print(rfm_df[['recency_days','purchase_frequency','monetary_total']].describe())

--- KIỂM CHỨNG RFM ---
Số khách hàng duy nhất (customer_unique_id): 93356
       recency_days  purchase_frequency  monetary_total
count  93356.000000         93356.00000    93356.000000
mean     237.973842             1.03342      141.404217
std      152.621155             0.20910      212.513508
min        1.000000             1.00000        0.000000
25%      114.000000             1.00000       47.650000
50%      219.000000             1.00000       89.700000
75%      346.000000             1.00000      154.532500
max      714.000000            15.00000    13440.000000


## Prepare for python

In [17]:
# --- Code gốc đã có trong báo cáo ---
items_clean = pd.read_sql("SELECT * FROM dbo.Cleaned_Orders_Items", engine)

df = (items_clean
     .merge(products[['product_id', 'product_category_name']], on='product_id')
     .merge(category, on='product_category_name', how='left'))

# --- Kiểm tra số dòng trước/sau merge (thêm mới) ---
print("Số dòng items_clean gốc:", items_clean.shape[0])
print("Số dòng sau merge với products:", df.shape[0])

# Kiểm tra số lượng NULL trước khi fillna
print("Số sản phẩm không có category (trước fillna):",
      df['product_category_name_english'].isna().sum())

df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

# Kiểm tra kết quả sau khi fill
print("Top 10 category phổ biến nhất sau khi merge:")
print(df['product_category_name_english'].value_counts().head(10))

C:\ProgramData\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1125.2'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Số dòng items_clean gốc: 112650
Số dòng sau merge với products: 112650
Số sản phẩm không có category (trước fillna): 1627
Top 10 category phổ biến nhất sau khi merge:
product_category_name_english
bed_bath_table           11115
health_beauty             9670
sports_leisure            8641
furniture_decor           8334
computers_accessories     7827
housewares                6964
watches_gifts             5991
telephony                 4545
garden_tools              4347
auto                      4235
Name: count, dtype: int64


In [18]:
# --- Code gốc đã có trong báo cáo ---
orders['hour'] = orders['order_purchase_timestamp'].dt.hour
orders['dayofweek'] = orders['order_purchase_timestamp'].dt.day_name()
orders['month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

heatmap_data = (orders.groupby(['dayofweek', 'hour']).size().reset_index(name='count'))
pivot = heatmap_data.pivot(index='dayofweek', columns='hour', values='count')

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex(day_order)

# --- Kiểm tra kết quả (thêm mới) ---
print("Kích thước bảng pivot (7 ngày x 24 giờ):", pivot.shape)
print("\n5 dòng đầu bảng pivot:")
print(pivot.head())

# Tìm khung giờ cao điểm nhất
peak_day, peak_hour = pivot.stack().idxmax()
peak_value = pivot.stack().max()
print(f"\nKhung giờ cao điểm nhất: {peak_day} lúc {peak_hour}h "
      f"với {peak_value} đơn hàng")

Kích thước bảng pivot (7 ngày x 24 giờ): (7, 24)

5 dòng đầu bảng pivot:
hour        0    1   2   3   4   5   6    7    8    9   ...    14    15    16  \
dayofweek                                               ...                     
Monday     328  134  66  36  21  22  69  160  479  783  ...  1096  1079  1094   
Tuesday    306  158  80  28  29  24  71  223  522  864  ...  1124  1047  1081   
Wednesday  397  179  81  33  33  27  93  211  517  829  ...  1050   983  1040   
Thursday   355  167  75  39  31  28  85  220  502  758  ...   977   928  1077   
Friday     426  216  72  49  40  36  97  206  493  768  ...   961   979   974   

hour        17   18   19    20    21   22   23  
dayofweek                                       
Monday     992  928  945  1027  1118  991  717  
Tuesday    967  877  924   988  1027  965  692  
Wednesday  967  852  848   904   963  878  615  
Thursday   909  784  826   839   840  857  551  
Friday     817  723  784   738   726  702  512  

[5 rows x 24 co

In [19]:
rfm = rfm_df.copy()

rfm['R'] = pd.qcut(rfm['recency_days'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['purchase_frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary_total'], q=5, labels=[1,2,3,4,5]).astype(int)

rfm['RFM_Score'] = rfm['R'] * 100 + rfm['F'] * 10 + rfm['M']

print(rfm.head(10))

                 customer_unique_id  recency_days  purchase_frequency  \
0  0000366f3b9a7992bf8c76cfdf3221e2           112                   1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f           115                   1   
2  0000f46a3911fa3c0805444483337064           537                   1   
3  0000f6ccb0745a6a4b88665a16c9f078           321                   1   
4  0004aac84e0df4da2b147fca70cf8255           288                   1   
5  0004bd2a26a76fe21f786e4fbd80607f           146                   1   
6  00050ab1314c0e55a6ca13cf7181fecf           132                   1   
7  00053a61a98854899e70ed204dd4bafe           183                   1   
8  0005e1862207bf6ccc02e4228effd9a0           543                   1   
9  0005ef4cd20d2893f0d9fbd94d3c0d97           170                   1   

   monetary_total  R  F  M  RFM_Score  
0          129.90  4  1  4        414  
1           18.90  4  1  1        411  
2           69.00  1  1  2        112  
3           25.99  2  1  1        21

### Python Data Analysis 

In [20]:
items_clean = pd.read_sql("SELECT * FROM dbo.Cleaned_Orders_Items", engine)
df = (items_clean
     .merge(products[['product_id', 'product_category_name']], on='product_id')
     .merge(category, on='product_category_name', how='left'))
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')


In [21]:
orders['hour'] = orders['order_purchase_timestamp'].dt.hour
orders['dayofweek'] = orders['order_purchase_timestamp'].dt.day_name()
orders['month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
heatmap_data = (orders.groupby(['dayofweek', 'hour']).size().reset_index(name='count'))
pivot = heatmap_data.pivot(index='dayofweek', columns='hour', values='count')
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex(day_order)


In [22]:
rfm = rfm_df.copy()
rfm['R'] = pd.qcut(rfm['recency_days'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['purchase_frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary_total'], q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R'] * 100 + rfm['F'] * 10 + rfm['M']
